# Investment AI
## Notebook 02 — Land Source Files into Unity Catalog Volumes

In Notebook 01, we created the governed Unity Catalog foundation.

Now we will move the source material from the Git repository into Unity Catalog Volumes.

The flow is:

```text
Git repository
│
├── data/documents
├── data/seed
└── data/semi_structured
        ↓
Unity Catalog Volumes
        ↓
Governed source storage
```

This notebook only **lands the files**.

We are not creating Delta tables, parsing documents, or building AI Search yet.


## What will be landed?

The repository contains three source groups:

```text
Research documents
→ PDF, DOCX, HTML, TXT

Structured seed data
→ CSV

Semi-structured data
→ Excel
```

We will intentionally **not** land:

- private evaluation ground truth under `evals/`
- the prompt-injection document under `security_test_documents/`

Those assets are reserved for later evaluation and security phases and must not leak into the production data path.


In [0]:
from pathlib import Path
from collections import Counter
import csv
import shutil

CATALOG = "investment_ai"

RESEARCH_SCHEMA = "research"
PORTFOLIO_SCHEMA = "portfolio"

RAW_DOCUMENTS_VOLUME = (
    f"/Volumes/{CATALOG}/{RESEARCH_SCHEMA}/raw_documents"
)

SEED_DATA_VOLUME = (
    f"/Volumes/{CATALOG}/{PORTFOLIO_SCHEMA}/seed_data"
)

SEMI_STRUCTURED_VOLUME = (
    f"/Volumes/{CATALOG}/{PORTFOLIO_SCHEMA}/semi_structured"
)

print("Research Volume        :", RAW_DOCUMENTS_VOLUME)
print("Portfolio Seed Volume  :", SEED_DATA_VOLUME)
print("Semi-structured Volume :", SEMI_STRUCTURED_VOLUME)


## 1. Locate the Git repository

Because this notebook lives inside the repository, we can find the repo root by walking upward until we find both:

- `pyproject.toml`
- `data/`

This avoids hard-coding a user-specific Databricks Workspace path.


In [0]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not find repository root. "
        "Run this notebook from inside the cloned Git repository."
    )


REPO_ROOT = find_repo_root(Path.cwd())

DOCUMENTS_SOURCE = REPO_ROOT / "data" / "documents"
SEED_SOURCE = REPO_ROOT / "data" / "seed"
SEMI_STRUCTURED_SOURCE = REPO_ROOT / "data" / "semi_structured"

print("Repository root :", REPO_ROOT)
print("Documents       :", DOCUMENTS_SOURCE)
print("Seed data       :", SEED_SOURCE)
print("Semi-structured :", SEMI_STRUCTURED_SOURCE)


## 2. Validate the source files before copying

Before moving anything into governed storage, we validate what is actually present in Git.

Expected research corpus:

- 45 documents
- 25 PDF
- 10 DOCX
- 5 HTML
- 5 TXT

We also expect:

- 6 CSV seed files
- 2 Excel files


In [0]:
research_files = sorted(
    p for p in DOCUMENTS_SOURCE.rglob("*")
    if p.is_file()
)

seed_files = sorted(
    p for p in SEED_SOURCE.glob("*")
    if p.is_file()
)

semi_structured_files = sorted(
    p for p in SEMI_STRUCTURED_SOURCE.glob("*")
    if p.is_file()
)

research_extension_counts = Counter(
    p.suffix.lower()
    for p in research_files
)

print("Research documents :", len(research_files))
print("By extension       :", dict(research_extension_counts))
print("Seed files         :", len(seed_files))
print("Excel files        :", len(semi_structured_files))

assert len(research_files) == 45
assert research_extension_counts == {
    ".pdf": 25,
    ".docx": 10,
    ".html": 5,
    ".txt": 5,
}
assert len(seed_files) == 6
assert len(semi_structured_files) == 2

print("\n✅ Repository source contract validated.")


## 3. Separate research metadata from portfolio seed data

One of the CSV files is `document_manifest.csv`.

That file describes the research corpus, so we will keep it with the research source area rather than mixing it with portfolio data.

The remaining seed files belong to the portfolio domain.


In [0]:
DOCUMENT_MANIFEST_SOURCE = (
    SEED_SOURCE / "document_manifest.csv"
)

portfolio_seed_files = [
    p for p in seed_files
    if p.name != "document_manifest.csv"
]

with DOCUMENT_MANIFEST_SOURCE.open(
    "r",
    encoding="utf-8",
    newline=""
) as handle:
    manifest_rows = list(csv.DictReader(handle))

print("Research manifest rows :", len(manifest_rows))
print("Portfolio CSV files    :", len(portfolio_seed_files))

assert len(manifest_rows) == 45
assert len(portfolio_seed_files) == 5

print("\nPortfolio seed files:")
for path in portfolio_seed_files:
    print(" -", path.name)

print("\n✅ Research metadata and portfolio seed data separated.")


## 4. Copy research documents to the research Volume

We preserve the company and quarter folder structure:

```text
raw_documents/
└── documents/
    ├── IA-SYN-01/Q2_2026/
    ├── IA-SYN-02/Q2_2026/
    ├── IA-SYN-03/Q2_2026/
    ├── IA-SYN-04/Q2_2026/
    └── IA-SYN-05/Q2_2026/
```

Preserving this structure makes the landed source easy to inspect before we begin document parsing.


In [0]:
RESEARCH_DOCUMENTS_TARGET = (
    Path(RAW_DOCUMENTS_VOLUME) / "documents"
)

for source in research_files:
    relative_path = source.relative_to(DOCUMENTS_SOURCE)
    target = RESEARCH_DOCUMENTS_TARGET / relative_path

    target.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(source, target)

print(
    f"✅ Landed {len(research_files)} research documents."
)


## 5. Land the research document manifest

The manifest is metadata for the research corpus, so we keep it in a separate `metadata` directory inside the research Volume.


In [0]:
RESEARCH_METADATA_TARGET = (
    Path(RAW_DOCUMENTS_VOLUME) / "metadata"
)

RESEARCH_METADATA_TARGET.mkdir(
    parents=True,
    exist_ok=True
)

manifest_target = (
    RESEARCH_METADATA_TARGET / "document_manifest.csv"
)

shutil.copy2(
    DOCUMENT_MANIFEST_SOURCE,
    manifest_target
)

print("✅ Landed research manifest:")
print(manifest_target)


## 6. Land structured portfolio seed data

These files will become governed Delta tables in the next stage of the project.


In [0]:
portfolio_seed_target = Path(SEED_DATA_VOLUME)

portfolio_seed_target.mkdir(
    parents=True,
    exist_ok=True
)

for source in portfolio_seed_files:
    shutil.copy2(
        source,
        portfolio_seed_target / source.name
    )

print(
    f"✅ Landed {len(portfolio_seed_files)} "
    "portfolio seed files."
)


## 7. Land semi-structured Excel data

We keep Excel sources separate from the CSV seed files because they will require a different ingestion path later.


In [0]:
semi_structured_target = Path(
    SEMI_STRUCTURED_VOLUME
)

semi_structured_target.mkdir(
    parents=True,
    exist_ok=True
)

for source in semi_structured_files:
    shutil.copy2(
        source,
        semi_structured_target / source.name
    )

print(
    f"✅ Landed {len(semi_structured_files)} "
    "semi-structured Excel files."
)


## 8. Verify the landed files

Now we validate the destination instead of assuming the copy succeeded.


In [0]:
landed_research_files = sorted(
    p for p in RESEARCH_DOCUMENTS_TARGET.rglob("*")
    if p.is_file()
)

landed_portfolio_files = sorted(
    p for p in Path(SEED_DATA_VOLUME).glob("*")
    if p.is_file()
)

landed_excel_files = sorted(
    p for p in Path(SEMI_STRUCTURED_VOLUME).glob("*")
    if p.is_file()
)

landed_research_extensions = Counter(
    p.suffix.lower()
    for p in landed_research_files
)

assert len(landed_research_files) == 45
assert landed_research_extensions == research_extension_counts
assert manifest_target.exists()
assert len(landed_portfolio_files) == 5
assert len(landed_excel_files) == 2

print("Research documents :", len(landed_research_files))
print("Research metadata  : 1")
print("Portfolio CSVs     :", len(landed_portfolio_files))
print("Excel sources      :", len(landed_excel_files))

print("\n✅ Destination validation passed.")


## 9. Confirm protected assets were not landed

Two areas must remain outside the production ingestion path:

```text
evals/private_ground_truth/
security_test_documents/
```

Why?

- evaluation ground truth should not be visible to the system we later evaluate
- the prompt-injection test file should not enter AI Search until the security phase

This prevents data leakage and keeps our later tests meaningful.


In [0]:
all_landed_names = {
    p.name
    for root in [
        Path(RAW_DOCUMENTS_VOLUME),
        Path(SEED_DATA_VOLUME),
        Path(SEMI_STRUCTURED_VOLUME),
    ]
    for p in root.rglob("*")
    if p.is_file()
}

assert (
    "DO_NOT_INDEX_UNTIL_SECURITY_PHASE_prompt_injection_test.txt"
    not in all_landed_names
)

assert "guidance_history.csv" not in all_landed_names
assert "company_fundamentals.csv" not in all_landed_names
assert "market_events.json" not in all_landed_names

print("✅ Private evaluation ground truth was not landed.")
print("✅ Prompt-injection test document was not landed.")


## 10. Source landing summary

Our governed source layer is now:

```text
/Volumes/investment_ai/research/raw_documents/
├── documents/
│   ├── IA-SYN-01/
│   ├── IA-SYN-02/
│   ├── IA-SYN-03/
│   ├── IA-SYN-04/
│   └── IA-SYN-05/
└── metadata/
    └── document_manifest.csv

/Volumes/investment_ai/portfolio/seed_data/
├── companies.csv
├── funds.csv
├── holdings.csv
├── prices.csv
└── benchmarks.csv

/Volumes/investment_ai/portfolio/semi_structured/
├── Investment_AI_Analyst_Estimates_Q2_2026.xlsx
└── Investment_AI_Portfolio_Risk_Q2_2026.xlsx
```

At this point the files are governed and landed, but they are still raw source files.

The next notebook will convert the structured portfolio sources into governed Delta tables.


In [0]:
print("✅ Source landing is complete.")
print("✅ 45 research documents landed.")
print("✅ Research manifest landed separately.")
print("✅ 5 portfolio CSV files landed.")
print("✅ 2 semi-structured Excel files landed.")
print("✅ Evaluation and security-only assets remain isolated.")
print("\nNext: 03_build_structured_delta_foundation.ipynb")
